In [0]:
from pyspark.sql import functions as F
import datetime as _dt

In [0]:
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

In [0]:
bookings = spark.table(f"{catalog}.bronze.bookings_inc").where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

current_dim = spark.sql(f"select customer_sk,customer_id from {catalog}.{schema}.customer_dim where is_current = true")

booking_enriched = (bookings.alias("t").join(current_dim.alias("s"),F.col("t.customer_id") == F.col("s.customer_id"),"left").withColumn("customer_sk",F.col("s.customer_sk")))

booking_enriched_sel= booking_enriched.select(F.col("booking_type"),F.col("t.customer_id").alias("customer_id"),F.col("customer_sk"),F.col("business_date"),F.col("amount"),F.col("discount"),F.col("quantity"))

agg = booking_enriched_sel.groupBy("booking_type","customer_sk","customer_id","business_date").agg(F.sum(F.col("amount")-F.col("discount")).alias("total_amount_sum"),F.sum("quantity").alias("total_quantity_sum"))

fact_full_name = f"{catalog}.{schema}.booking_fact"

spark.sql(f"create schema if not exists {catalog}.{schema}")

if not spark.catalog.tableExists(fact_full_name):
    a = agg.limit(0)
    a.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(fact_full_name)

agg.createOrReplaceTempView("src")
spark.sql(f"""merge into {fact_full_name} t using src s on t.customer_sk = s.customer_sk and t.booking_type <=> s.booking_type and t.business_date = s.business_date when matched then update set t.total_amount_sum = s.total_amount_sum, t.total_quantity_sum = s.total_quantity_sum when not matched then insert *""")
print("Fact build complete")